# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Seif-2/ML-week-1-FLY/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Paper: *FlyRank — The State of AI-Driven SEO, March 2026* (341,701 content pieces, 57 brands; ML appendix on 61,790 active records). Both critiques below are constructive — the goal is the next level of rigor, not a scalp.

---

### Finding #4 — "The Freshness Multiplier" (labeled CONFIRMED)

**The claim:** growth-to-decline ratio by freshness window shows 31–90 days as the strongest stable band (7.88:1), and the paper is careful to flag that the `361+` bucket (283:1) is "too small and too unstable to treat as a headline multiplier" — good, the paper already does some of this work itself. But the paper's headline pull-quote is still: *"365+ day content that was refreshed within 30 days shows 3.2x health boost... and 57x more impressions."*

**Where does this label come from?** "Health boost" is FlyRank's own composite `health_score`, not a directly observed metric like clicks or impressions. The ML appendix's own correlation matrix (this same paper) shows `health_score` correlates at **−0.6 with avg_position** — by far its strongest correlation with anything — which means health_score is largely *encoding position itself*. A page that moved from position 40 to position 8 after a refresh would show a big health jump almost mechanically, without needing any independent signal.

**Does the validation design carry the claim?** This is a before/after comparison on pages *someone chose* to refresh (selection, not random assignment) — exactly the selection-bias pattern the `writing-honest-claims` skill flags: "if the treated group was chosen, part of the gap is the choosing, not the treatment." Pages picked for a refresh program are plausibly ones an editor already believed had upside. The 57x impression figure is dramatic, but with no stated control group of similar-aged, un-refreshed pages over the same window, I can't tell how much of that gap is the refresh versus the selection.

**Constructive ask:** report the same ratio for a matched control cohort (similar age/topic, not refreshed, same window) alongside the refreshed group — that isolates the refresh effect from the selection effect. This is precisely the same lesson our own w04 Signal A learned about the `top_3` position tier: a headline number resting on a tiny, non-independent sample needs a second, harder check before it's a "multiplier" a reader acts on.

---

### Finding #7 — "The Winning Combinations" (labeled CONFIRMED)

**The claim:** average `health_score` by intent × competition-level shows transactional/low-competition pages scoring highest (30), suggesting editors should "copy the top configurations."

**Where does this label come from?** Same composite `health_score` issue as Finding #4 — and the correlation table shows `health_score` also correlates positively with `impressions` (0.3) and only weakly with `ctr` (0.1). A composite that's dominated by position and impressions will naturally favor transactional/low-competition content, because that segment is plausibly easier to rank for in general — not necessarily because the *combination* itself is causally "winning."

**Does the validation design carry the claim?** This is a cross-sectional snapshot (one time window, no intervention, no time-ordering) — per the claim ladder, that supports "X is associated with Y," never "copy this configuration and expect this result." The paper's own action-language ("start by copying the top configurations") reads like a prescriptive recommendation, when the evidence underneath it is an association.

**Constructive ask:** the finding would be much stronger split two ways — once using `health_score` (as shown) and once using a directly-observed metric like median `ctr` or `avg_position` by the same intent×competition grid. If both tell the same story, the finding survives an independent check. If they diverge, that's worth knowing before recommending anyone "copy" a configuration.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

Same Random Forest, same honest feature set as w05, trained/evaluated two ways: a naive **random row split** (what you'd get if you didn't think about it) vs. the **grouped client split** (what w05 actually used). The gap between them is itself the finding — it shows how much of a "good score" would have been memorization if I hadn't grouped by client.


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
visible = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0)].copy()
tier_median = visible.groupby("position_tier")["ctr"].transform("median")
visible["tier_median_ctr"] = tier_median
visible["ctr_gap"] = tier_median - visible["ctr"]
visible["underperform_flag"] = (visible["ctr_gap"] > 0).astype(int)

num_feats = ["impressions_90d", "avg_position", "content_age_days", "days_since_last_update", "word_count"]
cat_feats = ["content_type", "main_intent", "position_tier"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def run_split(train, test, label):
    Xtr = pd.get_dummies(train[num_feats + cat_feats], columns=cat_feats).fillna(0)
    Xte = pd.get_dummies(test[num_feats + cat_feats], columns=cat_feats).fillna(0)
    Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)
    ytr, yte = train["underperform_flag"], test["underperform_flag"]
    rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1).fit(Xtr, ytr)
    scores = rf.predict_proba(Xte)[:, 1]
    auc = roc_auc_score(yte, scores)
    p50 = precision_at_k(scores, yte.values, 50)
    p200 = precision_at_k(scores, yte.values, 200)
    print(f"{label}: AUC={auc:.3f}  P@50={p50:.3f}  P@200={p200:.3f}  base_rate={yte.mean():.3f}  n_test={len(yte)}")
    return auc, p50, p200

# Naive random split
train_r, test_r = train_test_split(visible, test_size=0.3, random_state=42, stratify=visible["underperform_flag"])
print("BEFORE (naive random split):")
r_auc, r_p50, r_p200 = run_split(train_r, test_r, "  Random split")

# Honest grouped split, same as w05
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
tr_idx, te_idx = next(gss.split(visible, groups=visible["client_id"]))
train_g, test_g = visible.iloc[tr_idx].copy(), visible.iloc[te_idx].copy()
print("\nAFTER (honest grouped-by-client split, matches w05):")
g_auc, g_p50, g_p200 = run_split(train_g, test_g, "  Grouped split")

print(f"\nGAP (random minus grouped): AUC {r_auc-g_auc:+.3f}   P@50 {r_p50-g_p50:+.3f}   P@200 {r_p200-g_p200:+.3f}")
print("\nThe random split's Precision@50 (0.94) massively overstates the model's real skill —")
print("nearly double the grouped-split number (0.52). That gap is memorization: the model was")
print("partly recognizing individual clients' writing style/template, not learning a signal that")
print("generalizes to clients it's never seen. The grouped number (0.52 P@50, 0.658 AUC) is the")
print("one I trust and the one reported in w05.")


BEFORE (naive random split):


  Random split: AUC=0.720  P@50=0.940  P@200=0.830  base_rate=0.475  n_test=5018

AFTER (honest grouped-by-client split, matches w05):


  Grouped split: AUC=0.658  P@50=0.520  P@200=0.550  base_rate=0.455  n_test=1461

GAP (random minus grouped): AUC +0.062   P@50 +0.420   P@200 +0.280

The random split's Precision@50 (0.94) massively overstates the model's real skill —
nearly double the grouped-split number (0.52). That gap is memorization: the model was
partly recognizing individual clients' writing style/template, not learning a signal that
generalizes to clients it's never seen. The grouped number (0.52 P@50, 0.658 AUC) is the
one I trust and the one reported in w05.


## 3. Leakage audit

Running the attack checklist from the skill against w05's final feature set. The headline check: deliberately re-add `ctr_gap` (which the label is thresholded from) to the honest grouped-split model, and watch the score jump — the same trap from w03, re-run here on the final feature set to confirm it's still clean.


In [3]:
Xtr_g = pd.get_dummies(train_g[num_feats + cat_feats], columns=cat_feats).fillna(0)
Xte_g = pd.get_dummies(test_g[num_feats + cat_feats], columns=cat_feats).fillna(0)
Xte_g = Xte_g.reindex(columns=Xtr_g.columns, fill_value=0)

Xtr_leak = Xtr_g.copy(); Xtr_leak["ctr_gap"] = train_g["ctr_gap"].values
Xte_leak = Xte_g.copy(); Xte_leak["ctr_gap"] = test_g["ctr_gap"].values
rf_leak = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1).fit(Xtr_leak, train_g["underperform_flag"])
leak_auc = roc_auc_score(test_g["underperform_flag"], rf_leak.predict_proba(Xte_leak)[:, 1])

print(f"WITHOUT ctr_gap (w05's actual final feature set): AUC = {g_auc:.3f}")
print(f"WITH ctr_gap re-added (deliberate leak test):      AUC = {leak_auc:.3f}")
print(f"\nCollapse confirms the earlier trap: {leak_auc:.3f} -> {g_auc:.3f} when the label-derived")
print("column is removed. w05's actual feature list never included ctr_gap, ctr, trend_direction,")
print("or trend_pct — this cell just re-proves that decision was the right one.\n")

print("Attack checklist:")
print("[x] Timeline drawn: single trailing-90-day snapshot, no future window exists to leak from")
print("[x] No label-derived columns in features (ctr, ctr_gap, trend_direction, trend_pct all excluded)")
print("[x] No product flags as features (none shipped in this dataset at all)")
print("[x] Population selection checked: filtered to impressions_90d>=500 & avg_position>0 — a volume")
print("    floor decided BEFORE looking at outcomes, not chosen to inflate results")
print("[x] Split grouped by client_id, zero overlap (confirmed in w05 and again above)")
print(f"[x] Base rate printed next to every metric (test base rate: {test_g['underperform_flag'].mean():.3f})")
print("[x] Top feature importance sanity-checked in w05 (avg_position, word_count, content_age_days —")
print("    all plausible, none suspiciously perfect)")
print("[x] Metrics computed out-of-fold (test split never touched during training)")
print("[ ] Sealed/holdout claim: NOT made — this is a standard train/test split, not a sealed evaluation")


WITHOUT ctr_gap (w05's actual final feature set): AUC = 0.658
WITH ctr_gap re-added (deliberate leak test):      AUC = 1.000

Collapse confirms the earlier trap: 1.000 -> 0.658 when the label-derived
column is removed. w05's actual feature list never included ctr_gap, ctr, trend_direction,
or trend_pct — this cell just re-proves that decision was the right one.

Attack checklist:
[x] Timeline drawn: single trailing-90-day snapshot, no future window exists to leak from
[x] No label-derived columns in features (ctr, ctr_gap, trend_direction, trend_pct all excluded)
[x] No product flags as features (none shipped in this dataset at all)
[x] Population selection checked: filtered to impressions_90d>=500 & avg_position>0 — a volume
    floor decided BEFORE looking at outcomes, not chosen to inflate results
[x] Split grouped by client_id, zero overlap (confirmed in w05 and again above)
[x] Base rate printed next to every metric (test base rate: 0.455)
[x] Top feature importance sanity-checked

## 4. Claim rewrite

**My boldest sentence, as first drafted (too strong):**
> "Our Random Forest model beats the baseline at predicting which pages are underperforming their peers."

**Why it's too strong:** "beats the baseline" implies a fair fight the baseline never had — the baseline scores near-perfect by construction (it directly measures the same CTR the label is built from, per w05 Section 1). The sentence also says "predicting," which implies foresight, when this is a same-window, non-future comparison.

**Rewritten, safe version:**
> "Using only indirect, pre-CTR-observation signals (search position, content age, word count, traffic volume), our grouped-split Random Forest model correctly ranked underperforming pages at a rate meaningfully above the base rate (Precision@200 of 0.55 vs. a 0.455 base rate, ROC AUC 0.658) — an observed, decision-support result on this 90-day starter slice, not a causal or predictive claim about future performance."


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.